# Mobile Plan Recommendation — Classification Model

## Project Overview

Megaline, a mobile carrier, wants to move subscribers away from legacy plans onto one of two newer plans: **Smart** or **Ultra**. This project builds a classification model that analyzes subscriber behavior data and recommends the most suitable plan for each user.

## Dataset

The dataset (`users_behavior.csv`) contains monthly behavior records for subscribers who have already switched to the new plans. Each row represents one subscriber with the following features:

| Feature | Description |
|---|---|
| `calls` | Number of calls made |
| `minutes` | Total call duration (minutes) |
| `messages` | Number of text messages sent |
| `mb_used` | Internet traffic used (MB) |
| `is_ultra` | Target: 1 = Ultra plan, 0 = Smart plan |

## Objective

Train and evaluate multiple classification models (Decision Tree, Random Forest, Logistic Regression) to predict whether a subscriber should be recommended the Ultra plan, targeting an accuracy threshold of **0.75**.

## 1. Data Exploration

In [8]:
import pandas as pd

df = pd.read_csv('datasets/users_behavior.csv')
print(df.info())
print(df.describe())
print(df['is_ultra'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None
             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269  17207.673836     0.306472
std      33.236368   234.569872    36.148326   7570.968246     0.461100
min       0.000000     0.000000     0.000000      0.000000     0.000000
25%      40.000000   274.575000     9.000000  12491.902500     0.000000
50%      62.000000   430.600000    30.000000  16943.235000     0.000000
75%      82.000000   571.927500    57.000000  21424.700000  

## 2. Train / Validation / Test Split

The dataset is split into three parts using a **60 / 20 / 20** ratio:
- **Training set** — used to fit each model
- **Validation set** — used to tune hyperparameters and select the best model
- **Test set** — held out for final, unbiased evaluation

In [9]:
from sklearn.model_selection import train_test_split

features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

# 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(features, target, test_size=0.4, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

## 3. Model Training & Hyperparameter Tuning

Three classifiers are trained and compared on the validation set.

### 3a. Decision Tree

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

best_tree = None
best_score = 0

for depth in range(1, 20):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    val_pred = model.predict(X_valid)
    score = accuracy_score(y_valid, val_pred)
    if score > best_score:
        best_score = score
        best_tree = model

print("Best Decision Tree Accuracy:", best_score)

Best Decision Tree Accuracy: 0.7962674961119751


### 3b. Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier

best_rf = None
best_score = 0

for est in [10, 50, 100]:
    for depth in [5, 10, 15]:
        model = RandomForestClassifier(n_estimators=est, max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        val_pred = model.predict(X_valid)
        score = accuracy_score(y_valid, val_pred)
        if score > best_score:
            best_score = score
            best_rf = model

print("Best Random Forest Accuracy:", best_score)

Best Random Forest Accuracy: 0.8133748055987559


### 3c. Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
val_pred = model.predict(X_valid)
print("Logistic Regression Accuracy:", accuracy_score(y_valid, val_pred))

Logistic Regression Accuracy: 0.7402799377916018


## 4. Final Model Evaluation

Random Forest achieved the highest validation accuracy (**81.3%**) and is selected as the final model. It is now evaluated on the held-out test set.

In [13]:
test_pred = best_rf.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)
print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8102643856920684


In [14]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred))

[[410  38]
 [ 84 111]]
              precision    recall  f1-score   support

           0       0.83      0.92      0.87       448
           1       0.74      0.57      0.65       195

    accuracy                           0.81       643
   macro avg       0.79      0.74      0.76       643
weighted avg       0.80      0.81      0.80       643



## 5. Conclusion

The **Random Forest classifier** was selected as the best-performing model with a test accuracy of **81.0%**, well above the 75% threshold.

| Model | Validation Accuracy |
|---|---|
| Decision Tree | 79.6% |
| **Random Forest** | **81.3%** |
| Logistic Regression | 74.0% |

The model performs well at identifying Smart plan users (precision 0.83, recall 0.92) and shows reasonable performance for Ultra plan users (precision 0.74, recall 0.57). Further improvements could be explored through additional feature engineering or ensemble tuning.